In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/processed/smartblood_dataset_2023_2024_eda_ready.csv")

df["date"] = pd.to_datetime(df["date"])

df.head()

,date,blood_group,component,demand,collected,issued,opening_stock,closing_stock,expired,emergency,...,stock_difference,expiry_rate,shortage_risk,stock_coverage_ratio,excess_stock_risk,expiry_risk,emergency_risk,demand_lag_1,demand_lag_7,demand_rolling_7
0,2023-01-01,A+,Plasma,32,31,31,102,102,0,1,...,0,0.000000,0,3.090909,1,0,1,NaN,NaN,NaN
1,2023-01-02,A+,Plasma,35,38,36,102,104,0,9,...,0,0.000000,0,2.888889,1,0,1,32.0,NaN,NaN
2,2023-01-03,A+,Plasma,23,18,23,104,99,0,0,...,0,0.000000,0,4.125000,1,0,0,35.0,NaN,NaN
3,2023-01-04,A+,Plasma,52,48,51,99,95,1,12,...,0,0.006757,0,1.792453,0,1,1,23.0,NaN,NaN
4,2023-01-05,A+,Plasma,32,26,29,95,92,0,2,...,0,0.000000,0,2.787879,1,0,1,52.0,NaN,NaN


In [2]:
df = df.sort_values(
    ["blood_group", "component", "date"]
).reset_index(drop=True)

In [3]:
df = df.sort_values(["blood_group", "component", "date"]).reset_index(drop=True)

df["day_of_week"] = df["date"].dt.day_name()
df["day_of_week_num"] = df["date"].dt.dayofweek
df["month"] = df["date"].dt.month
df["month_name"] = df["date"].dt.month_name()
df["year"] = df["date"].dt.year
df["quarter"] = df["date"].dt.quarter
df["is_weekend"] = (df["day_of_week_num"] >= 5).astype(int)

df["blood_product"] = df["blood_group"] + " - " + df["component"]

df.head()

,date,blood_group,component,demand,collected,issued,opening_stock,closing_stock,expired,emergency,...,stock_difference,expiry_rate,shortage_risk,stock_coverage_ratio,excess_stock_risk,expiry_risk,emergency_risk,demand_lag_1,demand_lag_7,demand_rolling_7
0,2023-01-01,A+,Plasma,32,31,31,102,102,0,1,...,0,0.000000,0,3.090909,1,0,1,NaN,NaN,NaN
1,2023-01-02,A+,Plasma,35,38,36,102,104,0,9,...,0,0.000000,0,2.888889,1,0,1,32.0,NaN,NaN
2,2023-01-03,A+,Plasma,23,18,23,104,99,0,0,...,0,0.000000,0,4.125000,1,0,0,35.0,NaN,NaN
3,2023-01-04,A+,Plasma,52,48,51,99,95,1,12,...,0,0.006757,0,1.792453,0,1,1,23.0,NaN,NaN
4,2023-01-05,A+,Plasma,32,26,29,95,92,0,2,...,0,0.000000,0,2.787879,1,0,1,52.0,NaN,NaN


In [4]:
group_cols = ["blood_group", "component"]

df["demand_lag_1"] = (
    df.groupby(group_cols)["demand"].shift(1)
)

df["demand_lag_7"] = (
    df.groupby(group_cols)["demand"].shift(7)
)

df["demand_lag_14"] = (
    df.groupby(group_cols)["demand"].shift(14)
)

df["demand_lag_30"] = (
    df.groupby(group_cols)["demand"].shift(30)
)

In [5]:
df["demand_rolling_7"] = (
    df.groupby(group_cols)["demand"]
      .transform(lambda s: s.shift(1).rolling(7).mean())
)

df["demand_rolling_14"] = (
    df.groupby(group_cols)["demand"]
      .transform(lambda s: s.shift(1).rolling(14).mean())
)

df["demand_rolling_30"] = (
    df.groupby(group_cols)["demand"]
      .transform(lambda s: s.shift(1).rolling(30).mean())
)

In [6]:
df["emergency_requests_lag_1"] = (
    df.groupby(group_cols)["emergency"]
      .shift(1)
)

df["target_demand"] = (
    df.groupby(group_cols)["demand"]
    .shift(-1)
)

In [7]:
df[["date", "blood_group", "component", "demand", "demand_lag_1", "demand_lag_7", "demand_lag_14", "demand_lag_30", "demand_rolling_7", "demand_rolling_14", "demand_rolling_30", "emergency", "emergency_requests_lag_1", "target_demand"]].head(40)

,date,blood_group,component,demand,demand_lag_1,demand_lag_7,demand_lag_14,demand_lag_30,demand_rolling_7,demand_rolling_14,demand_rolling_30,emergency,emergency_requests_lag_1,target_demand
0,2023-01-01,A+,Plasma,32,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,NaN,35.0
1,2023-01-02,A+,Plasma,35,32.0,NaN,NaN,NaN,NaN,NaN,NaN,9,1.0,23.0
2,2023-01-03,A+,Plasma,23,35.0,NaN,NaN,NaN,NaN,NaN,NaN,0,9.0,52.0
3,2023-01-04,A+,Plasma,52,23.0,NaN,NaN,NaN,NaN,NaN,NaN,12,0.0,32.0
4,2023-01-05,A+,Plasma,32,52.0,NaN,NaN,NaN,NaN,NaN,NaN,2,12.0,30.0
5,2023-01-06,A+,Plasma,30,32.0,NaN,NaN,NaN,NaN,NaN,NaN,3,2.0,36.0
6,2023-01-07,A+,Plasma,36,30.0,NaN,NaN,NaN,NaN,NaN,NaN,4,3.0,31.0
7,2023-01-08,A+,Plasma,31,36.0,32.0,NaN,NaN,34.285714,NaN,NaN,2,4.0,29.0
8,2023-01-09,A+,Plasma,29,31.0,35.0,NaN,NaN,34.142857,NaN,NaN,1,2.0,32.0
9,2023-01-10,A+,Plasma,32,29.0,23.0,NaN,NaN,33.285714,NaN,NaN,4,1.0,26.0


## df
##  ↓
## original cleaned dataset

## df_ml
##  ↓
## ML-ready dataset

In [8]:
df.isnull().sum()

date                          0
blood_group                   0
component                     0
demand                        0
collected                     0
issued                        0
opening_stock                 0
closing_stock                 0
expired                       0
emergency                     0
day_of_week                   0
day_of_week_num               0
month                         0
month_name                    0
year                          0
quarter                       0
is_weekend                    0
blood_product                 0
calculated_closing_stock      0
stock_difference              0
expiry_rate                   0
shortage_risk                 0
stock_coverage_ratio          0
excess_stock_risk             0
expiry_risk                   0
emergency_risk                0
demand_lag_1                 24
demand_lag_7                168
demand_rolling_7            168
demand_lag_14               336
demand_lag_30               720
demand_r

In [9]:
df_ml = df.dropna().copy()
df_ml.isnull().sum()

date                        0
blood_group                 0
component                   0
demand                      0
collected                   0
issued                      0
opening_stock               0
closing_stock               0
expired                     0
emergency                   0
day_of_week                 0
day_of_week_num             0
month                       0
month_name                  0
year                        0
quarter                     0
is_weekend                  0
blood_product               0
calculated_closing_stock    0
stock_difference            0
expiry_rate                 0
shortage_risk               0
stock_coverage_ratio        0
excess_stock_risk           0
expiry_risk                 0
emergency_risk              0
demand_lag_1                0
demand_lag_7                0
demand_rolling_7            0
demand_lag_14               0
demand_lag_30               0
demand_rolling_14           0
demand_rolling_30           0
emergency_

In [10]:
print("Original rows:", len(df))
print("ML rows:", len(df_ml))
print("Rows removed:", len(df) - len(df_ml))

Original rows: 17544
ML rows: 16800
Rows removed: 744


In [11]:
df_ml.groupby(
    ["blood_group", "component"]
).size()

blood_group  component
A+           Plasma       700
             Platelets    700
             RBC          700
A-           Plasma       700
             Platelets    700
             RBC          700
AB+          Plasma       700
             Platelets    700
             RBC          700
AB-          Plasma       700
             Platelets    700
             RBC          700
B+           Plasma       700
             Platelets    700
             RBC          700
B-           Plasma       700
             Platelets    700
             RBC          700
O+           Plasma       700
             Platelets    700
             RBC          700
O-           Plasma       700
             Platelets    700
             RBC          700
dtype: int64

## data/
## ├── raw/
## ├── processed/
## └── features/

In [12]:
df_ml.to_csv(
    "../data/features/blood_demand_features.csv",
    index=False
)

In [13]:
df_ml["naive_prediction"] = df_ml["demand_lag_1"]

In [14]:
df_ml.naive_prediction

30       23.0
31       29.0
32       30.0
33       25.0
34       26.0
         ... 
17538    11.0
17539    15.0
17540    18.0
17541    18.0
17542    15.0
Name: naive_prediction, Length: 16800, dtype: float64

In [15]:
df_ml["moving_average_prediction"] = (
    df_ml["demand_rolling_7"]
)

In [16]:
df_ml.moving_average_prediction

30       30.000000
31       30.142857
32       30.142857
33       29.142857
34       28.857143
           ...    
17538    12.571429
17539    10.714286
17540    12.142857
17541    13.142857
17542    13.714286
Name: moving_average_prediction, Length: 16800, dtype: float64

# baseline

In [28]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error
)

In [30]:
# Naive Prediction
mae_naive = mean_absolute_error(
    test["target_demand"],
    test["naive_prediction"]
)

rmse_naive = np.sqrt(mean_squared_error(
    test["target_demand"],
    test["naive_prediction"]
))

In [31]:
df = pd.read_csv(
    "../data/features/blood_demand_features.csv"
)

In [32]:
df["date"] = pd.to_datetime(df["date"])

In [33]:
split_date = "2024-08-01"

train = df[df["date"] < split_date].copy()

test = df[df["date"] >= split_date].copy()

print("Train shape:", train.shape)
print("Test shape:", test.shape)

print("\nTrain date range:")
print(train["date"].min(), "→", train["date"].max())

print("\nTest date range:")
print(test["date"].min(), "→", test["date"].max())

Train shape: (13152, 35)
Test shape: (3648, 35)

Train date range:
2023-01-31 00:00:00 → 2024-07-31 00:00:00

Test date range:
2024-08-01 00:00:00 → 2024-12-30 00:00:00


In [34]:
# Tomorrow's prediction = today's demand
test["naive_prediction"] = test["demand_lag_1"]

In [35]:
# Tomorrow's prediction =
# average demand of previous 7 days
test["ma7_prediction"] = test["demand_rolling_7"]

In [36]:
mae_naive = mean_absolute_error(
    test["target_demand"],
    test["naive_prediction"]
)

rmse_naive = np.sqrt(
    mean_squared_error(
        test["target_demand"],
        test["naive_prediction"]
    )
)


mae_ma7 = mean_absolute_error(
    test["target_demand"],
    test["ma7_prediction"]
)

rmse_ma7 = np.sqrt(mean_squared_error(
    test["target_demand"],
    test["ma7_prediction"]
))

In [37]:
print("\n==============================")
print("BASELINE MODEL RESULTS")
print("==============================")

print("\nNaive Forecast")
print("MAE :", mae_naive)
print("RMSE:", rmse_naive)

print("\n7-Day Moving Average")
print("MAE :", mae_ma7)
print("RMSE:", rmse_ma7)


BASELINE MODEL RESULTS

Naive Forecast
MAE : 5.874451754385965
RMSE: 7.752051228263009

7-Day Moving Average
MAE : 4.366306390977443
RMSE: 5.753201019052642


In [38]:
results = pd.DataFrame({
    "Model": [
        "Naive Forecast",
        "7-Day Moving Average"
    ],

    "MAE": [
        mae_naive,
        mae_ma7
    ],

    "RMSE": [
        rmse_naive,
        rmse_ma7
    ]
})

print("\n==============================")
print("COMPARISON TABLE")
print("==============================")

display(results)


COMPARISON TABLE


,Model,MAE,RMSE
0,Naive Forecast,5.874452,7.752051
1,7-Day Moving Average,4.366306,5.753201


In [39]:
results.to_csv(
    "../outputs/results/baseline_results.csv",
    index=False
)

print("\nBaseline results saved successfully.")


Baseline results saved successfully.
